# Verify the shipped AASIST ONNX graph
The model is already exported upstream and pinned in `packages/audio-detector/assets/`. Do not re-export a different checkpoint here: this notebook checks the bundled graph's dimensions, opset, and raw input against the recorded metadata.

In [ ]:
!pip -q install onnx==1.19.0
from pathlib import Path
import json, onnx
root = Path('../..')
model_path = root / 'packages/audio-detector/assets/model.onnx'
meta = json.loads((root / 'packages/audio-detector/assets/meta.json').read_text())
model = onnx.load(str(model_path)); onnx.checker.check_model(model)
tensor = lambda value: (value.name, [d.dim_value or d.dim_param for d in value.type.tensor_type.shape.dim])
print('bytes:', model_path.stat().st_size)
print('input:', tensor(model.graph.input[0]))
print('output:', tensor(model.graph.output[0]))
print('opsets:', [(op.domain, op.version) for op in model.opset_import])
assert tensor(model.graph.input[0]) == ('wav', ['batch', 64600])
assert tensor(model.graph.output[0]) == ('logits', ['batch', 2])